# construction of Mouse tissue-specific metabolic network using CORDA in python 
# tutorial avaiable at (https://resendislab.github.io/corda/)
# repo source (https://github.com/resendislab/corda)

In order to start a reconstruction CORDA requires you to assign a confidence score to each reaction in your base model. This can be done by assigning confidence based on proteome or gene expression data.

CORDA manages a total of 5 confidence levels:

-1 for reactions that are not present and should not be included in the model
0 for reactions with unknown confidence which may be included in the model if necessary
1 for low confidence reactions that should be included if necessary
2 for medium confidence reactions that should be included if necessary
3 for high confidence reactions that must be included if possible in any way

In [ ]:
import pandas as pd
import cobra
import os 
from cobra.io import read_sbml_model
from cobra.util.solver import solvers
from corda import CORDA

In [ ]:
# read the reactions confidences assigned from MATLAb 
# Read the reaction scores file
reaction_scores_df = pd.read_csv('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/CORDA_mouse_models/input_data/brain_rxns_scores.csv', index_col=0)
print(reaction_scores_df.head())  # Check the first few rows

# Convert to a dictionary
reaction_confidences = reaction_scores_df['CORDA_Score'].to_dict()

In [ ]:
# read the mouse comprehensive GEM iMM1865 (xml version)
# Set Gurobi as the solver
cobra.Configuration().solver = "gurobi"

# Confirm the selected solver
print("Selected Solver:", cobra.Configuration().solver)

# Load your model
model = cobra.io.read_sbml_model('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/mouse_model_iMM1865/iMM1865.xml')

# Test optimization
solution = model.optimize()
print("Objective Value:", solution.objective_value)

In [ ]:
# reconstruct the tissue-specific model using CORDA
opt = CORDA(model, reaction_confidences, n=1)
opt.build()
print(opt)

In [ ]:
# set the objective function of the reconstructed model as that of iMM1865 (BIOMASS_reaction)
tissue_specific_model = opt.cobra_model()
tissue_specific_model.objective = tissue_specific_model.reactions.get_by_id("BIOMASS_reaction") #assign the objective function
tissue_specific_model.id = "CORDA_iMiceBrain" #assign a new name for the reconstruction

In [ ]:
# inspect the objective function of the tissue-specific reconstruction
print(tissue_specific_model.objective)

In [ ]:
# insepect the tissue-specific reconstruction genes and reactions 
print(len(tissue_specific_model.reactions))
print(len(tissue_specific_model.genes))

In [ ]:
# export the tissue-specific reconstruction as xml version
# export the reconstructed  model as SBML (Cobrapy compatible)
# Systems Biology Markup Language is an XML-based standard format for distributing models 
from cobra.io import write_sbml_model, validate_sbml_model
from pprint import pprint

# Define the file path for saving and validation
file_path = "/Users/eso1993/Desktop/CORDA_iMiceBrain.xml"

# Write the reconstructed model to the SBML file
write_sbml_model(tissue_specific_model, filename=file_path)

# Validate the SBML file
report = validate_sbml_model(filename=file_path)

# Display the validation report
pprint(report)

In [ ]:
# export the tissue-specific reconstruction as matlab version
import numpy as np
from scipy.io import savemat
from pathlib import Path
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
import logging

save_matlab_model(tissue_specific_model, "/Users/eso1993/Desktop/CORDA_iMiceBrain.mat")